# CLASIFICACIÓN DE CORREOS EN SPAM/NO SPAM

In [ ]:
import pandas as pd

# Lista  de columnas según el dataset de Spambase (57 características + 1 objetivo)
column_names = [
    "word_freq_make", "word_freq_address", "word_freq_all", "word_freq_3d", 
    "word_freq_our", "word_freq_over", "word_freq_remove", "word_freq_internet", 
    "word_freq_order", "word_freq_mail", "word_freq_receive", "word_freq_will", 
    "word_freq_people", "word_freq_report", "word_freq_addresses", "word_freq_free", 
    "word_freq_business", "word_freq_email", "word_freq_you", "word_freq_credit", 
    "word_freq_your", "word_freq_font", "word_freq_000", "word_freq_money", 
    "word_freq_hp", "word_freq_hpl", "word_freq_george", "word_freq_650", 
    "word_freq_lab", "word_freq_labs", "word_freq_telnet", "word_freq_857", 
    "word_freq_data", "word_freq_415", "word_freq_85", "word_freq_technology", 
    "word_freq_1999", "word_freq_parts", "word_freq_pm", "word_freq_direct", 
    "word_freq_cs", "word_freq_meeting", "word_freq_original", "word_freq_project", 
    "word_freq_re", "word_freq_edu", "word_freq_table", "word_freq_conference", 
    "char_freq_;", "char_freq_(", "char_freq_[", "char_freq_!", "char_freq_$", 
    "char_freq_#", "capital_run_length_average", "capital_run_length_longest", 
    "capital_run_length_total", "is_spam" # Añadimos la columna objetivo
]

# Cargar el dataset
df = pd.read_csv('spambase.data', names=column_names)


## Clasificación utilizando Random Forrest

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# X son las características (todas menos la última)
# y es el objetivo (la columna is_spam)
X = df.drop('is_spam', axis=1)
y = df['is_spam']

# 2. Dividir en set de Entrenamiento (80%) y Prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Crear y entrenar el modelo
# Usamos Random Forest con 100 árboles
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 4. Realizar predicciones
y_pred = model.predict(X_test)

# 5. Evaluar los resultados
print("--- Reporte de Clasificación ---")
print(classification_report(y_test, y_pred))

print(f"Precisión Global (Accuracy): {accuracy_score(y_test, y_pred):.2%}")

--- Reporte de Clasificación ---
              precision    recall  f1-score   support

           0       0.94      0.98      0.96       531
           1       0.98      0.92      0.95       390

    accuracy                           0.96       921
   macro avg       0.96      0.95      0.95       921
weighted avg       0.96      0.96      0.96       921

Precisión Global (Accuracy): 95.55%


Vemos un gran rendimiento del clasificador por árbol de regresión.
* El modelo tiene una Precision de 0.98 para la clase 1. Esto significa que de cada 100 correos que el modelo marcó como "Spam", 98 realmente lo eran.
* El Recall en el Spam es 0.92, lo que significa que de todos los correos basura que llegaron, el modelo detectó el 92%. Esto quiere decir que hay un 8% de spam que se está "colando" en la bandeja de entrada. Es un margen aceptable, pero es donde el modelo tiene más espacio para mejorar.
* El Recall de 0.98 para los correos buenos significa que casi ningún correo legítimo se está perdiendo.
* El F1-score dio0.96 y 0.95. Tener valores por encima de 0.95 en ambas clases indica que el clasificador es robusto y equilibrado. No sacrificó una métrica para inflar la otra.

In [6]:
# Ver las 10 características más importantes
importances = pd.Series(model.feature_importances_, index=X.columns)
print("\nTop 10 características más importantes:")
print(importances.sort_values(ascending=False).head(10))


Top 10 características más importantes:
char_freq_!                   0.113763
char_freq_$                   0.096754
word_freq_remove              0.081876
word_freq_free                0.067147
capital_run_length_longest    0.058521
capital_run_length_average    0.057862
capital_run_length_total      0.052362
word_freq_your                0.046289
word_freq_hp                  0.042406
word_freq_you                 0.032907
dtype: float64


Como era de esperarse, los caracteres que tienen mas insidencia en la clasificación de spam son "!" y "$". Las palabras como "gratis" y "tu" también parecen ser muy comunes en los mails de spam

Ahora crearemos dos funciones:
* `extract_features(text)`: convierte texto crudo en vector de 57 features.
* `predict_spam(text)`: devuelve `"SPAM"` o `"NOT SPAM"`

La idea es luego redactar un mail inventado, pasarlo por la función de `predict_spam()`, y que muestre el resultado de la clasificación

In [20]:
import re
import numpy as np
import pandas as pd

def extract_features(text, column_names):
    """
    Transforma un texto crudo en un vector de 57 características 
    compatibles con el dataset spambase.
    """
    # 1. Limpieza básica y conteo de palabras/caracteres
    words = re.findall(r'\b\w+\b', text.lower())
    total_words = len(words) if len(words) > 0 else 1
    total_chars = len(text) if len(text) > 0 else 1
    
    # Lista de palabras objetivo (las primeras 48 del dataset)
    target_words = [name.replace('word_freq_', '') for name in column_names[:48]]
    # Lista de caracteres objetivo (los siguientes 6)
    target_chars = [';', '(', '[', '!', '$', '#']
    
    features = []
    
    # Calcular frecuencias de palabras (%)
    for word in target_words:
        count = words.count(word)
        features.append((count / total_words) * 100)
        
    # Calcular frecuencias de caracteres (%)
    for char in target_chars:
        count = text.count(char)
        features.append((count / total_chars) * 100)
        
    # Estadísticas de Mayúsculas (Capital Run Length)
    cap_runs = re.findall(r'[A-Z]+', text)
    if cap_runs:
        run_lengths = [len(run) for run in cap_runs]
        features.append(np.mean(run_lengths)) # average
        features.append(max(run_lengths))     # longest
        features.append(sum(run_lengths))     # total
    else:
        features.extend([0, 0, 0])
        
    return np.array(features).reshape(1, -1)

# Definir la función de predicción
def predict_spam(text):
    features = extract_features(text, column_names[:-1])  # Excluimos 'is_spam'
    prediction = model.predict(features)
    probability = model.predict_proba(features)
    label = "SPAM" if prediction[0] == 1 else "NOT SPAM"
    confidence = probability[0][prediction[0]] * 100
    return f"{label} (probabilidad: {confidence:.1f}%)"

Una vez creadas dichas funciones, se procederá con los ejemplos

In [21]:
import warnings
warnings.filterwarnings('ignore')

nuevo_email = """
SUBJECT: URGENT BUSINESS PROPOSAL
FREE MONEY NOW! Click here to receive your credit today. 
Do not miss this limited technology offer !!! $$$
"""

print(f"Predicción para el nuevo email: {predict_spam(nuevo_email)}")

Predicción para el nuevo email: SPAM (probabilidad: 83.5%)


In [22]:
nuevo_email = """
SUBJECT: Question about your product
I am interested in learning more about the IPhone 15 you are offering, I saw it on your website. 
Could you please provide more details and pricing information?
"""
print(f"Predicción para el nuevo email: {predict_spam(nuevo_email)}")

Predicción para el nuevo email: NOT SPAM (probabilidad: 59.0%)
